In [24]:
import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np, kagglehub, random
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms, models
from aijack.defense import GeneralMomentAccountant, DPSGDManager
from tqdm import tqdm

# --- 1. RIPRODUCIBILITÀ ---
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    print(f"✅ Seed impostato a: {seed}")

set_seed(42)

# --- 2. PARAMETRI ---
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
batch_size = 16
epochs = 5
lr = 0.001
sigma = 0.8
l2_norm_clip = 1.0

# --- 3. DATI ---
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# --- 4. CREAZIONE SUBSET (FISSO GRAZIE AL SEED) ---
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)
subset_indices = indices[:1200]
small_dataset = Subset(full_dataset, subset_indices)

# Loader per il PrivacyManager
client_loader = DataLoader(small_dataset, batch_size=batch_size, shuffle=True)

print(f"🚀 Dataset pronto: {len(small_dataset)} immagini. Device: {device}")

✅ Seed impostato a: 42
🚀 Dataset pronto: 1200 immagini. Device: mps


In [25]:
def get_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    for param in model.conv1.parameters(): param.requires_grad = True
    
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    for param in model.fc.parameters(): param.requires_grad = True
    
    return model.to(device)

global_model = get_model(len(full_dataset.classes))

In [31]:
accountant = GeneralMomentAccountant(noise_type="Gaussian", backend="python")
iterations = epochs * (len(small_dataset) // batch_size)

privacy_manager = DPSGDManager(
    accountant, optim.Adam, l2_norm_clip=l2_norm_clip,
    dataset=small_dataset, lot_size=batch_size, batch_size=batch_size,
    iterations=iterations
)

# Filtriamo i parametri
trainable_params = [p for p in global_model.parameters() if p.requires_grad]
dpopt_cls, lot_loader, batch_loader = privacy_manager.privatize(noise_multiplier=sigma)
optimizer = dpopt_cls(trainable_params, lr=lr)

# --- FORZATURA DEGLI ACCUMULATORI ---
# Questo risolve l'AttributeError inizializzando i buffer interni di AIJack
for group in optimizer.param_groups:
    group["accum_grads"] = [torch.zeros_like(p.data) if p.requires_grad else None for p in group["params"]]
    for p in group["params"]:
        if p.requires_grad:
            p.grad = torch.zeros_like(p.data)

print("🛡️ Buffer AIJack forzati a Zero. L'errore dovrebbe sparire.")

🛡️ Buffer AIJack forzati a Zero. L'errore dovrebbe sparire.


In [32]:
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    correct, total = 0, 0
    global_model.train()
    
    # AIJack vuole gestire il loop del lotto
    pbar = tqdm(lot_loader(optimizer), desc=f"Epoca {epoch+1}/{epochs}")
    
    for X_lot, y_lot in pbar:
        # 1. Reset degli accumulatori del lotto
        for group in optimizer.param_groups:
            for accum_grad in group["accum_grads"]:
                if accum_grad is not None:
                    accum_grad.zero_()

        # 2. Loop sui micro-batch
        lot_dataset = TensorDataset(X_lot, y_lot)
        for X_batch, y_batch in batch_loader(lot_dataset):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = global_model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            
            # Clipping e accumulo manuale nel buffer del lotto
            optimizer.accumulate_grad()
            
            # Metriche
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
            
        # Lo step del lotto (che usa gli accum_grads che abbiamo appena riempito)
        # viene chiamato automaticamente dal lot_loader alla fine di questa iterazione
        
        if total > 0:
            acc = 100. * correct / total
            eps = accountant.get_epsilon(delta=1e-5)
            pbar.set_postfix({"Acc": f"{acc:.1f}%", "Eps": f"{eps:.2f}"})

print(f"🏁 Finalmente! Acc: {acc:.2f}%")

Epoca 5/5: 100%|██████████| 375/375 [00:45<00:00,  8.27it/s, Acc=41.4%, Eps=6.34]

🏁 Finalmente! Acc: 41.44%
